# BloomBERT — Tuned Run

Targeted hyperparameter tuning over the baseline (`train_bloombert.ipynb`).
Goal: push macro-F1 past the baseline's 0.7261 by attacking the easy↔medium and medium↔hard boundary confusion observed in the baseline confusion matrix.

## What's different from the baseline

| Change | Baseline | Tuned | Why |
|---|---|---|---|
| Loss function | class-weighted cross-entropy | **Focal loss (γ=2)** + class weights | down-weights easy examples, focuses gradient on uncertain boundary cases |
| Label smoothing | none | **0.1** | reduces overconfidence on noisy boundary labels |
| Pooling | [CLS] token (default DistilBertForSequenceClassification) | **mean-pool over attention-masked tokens** | longer hard questions retain more discriminative signal |
| LR schedule | linear warmup + linear decay | **linear warmup + cosine decay** | sharper late-stage refinement |
| Epochs | 4 | **6** | cosine benefits from longer schedule |
| Unfrozen layers | (2 or 3) | **3** | best from baseline ablation |
| Top-layer dropout | 0.1 | **0.2** | combats mild epoch-4 overfit observed in baseline |

Everything else (LR=2e-5, LLRD=0.9, fp16, batch=32, max_len=128, seed=42) is identical to baseline so comparisons are clean.

## Setup (same as baseline)
1. Attach the `baylearn-bloom-data` Kaggle dataset (Add Input panel).
2. Accelerator → GPU T4 x1.
3. Run All. ~25 min on T4.
4. Output: `/kaggle/working/bloom_distilbert_tuned/`. Download and extract to `models/bloom_distilbert/` (overwriting baseline) if results are better.

In [ ]:
# ---- Cell 1: Imports + Config -------------------------------------------
import os, json, random, math, csv
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    DistilBertTokenizerFast,
    DistilBertModel,        # bare encoder; we plug in mean-pool ourselves
    DistilBertConfig,
    get_cosine_schedule_with_warmup,
)
from sklearn.metrics import classification_report, confusion_matrix, f1_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

Config = {
    'model_name'    : 'distilbert-base-uncased',
    'max_len'       : 128,
    'batch_size'    : 32,
    'epochs'        : 8,           # ↑ from 6 (per user request: 8 for all non-baseline)
    'lr'            : 2e-5,
    'llrd_decay'    : 0.9,
    'weight_decay'  : 0.01,
    'warmup_frac'   : 0.10,
    'unfreeze_top'  : 3,
    'num_classes'   : 3,
    'use_amp'       : True,
    # tuning bundle
    'focal_gamma'   : 2.0,
    'label_smoothing': 0.1,
    'pooling'       : 'mean',
    'head_dropout'  : 0.2,
}
LABEL_TO_ID = {'easy': 0, 'medium': 1, 'hard': 2}
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}

DATA_DIR = '/kaggle/input/datasets/manarabdelshafy/baylearn-bloom-data'
OUT_DIR  = '/kaggle/working/bloom_distilbert_tuned'
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'Epochs: {Config["epochs"]}   Focal γ: {Config["focal_gamma"]}   '
          f'Label smoothing: {Config["label_smoothing"]}   Pooling: {Config["pooling"]}')

In [ ]:
# ---- Cell 2: Data (identical to baseline) ---------------------------------
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
val_df   = pd.read_csv(f'{DATA_DIR}/val.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    before = len(df)
    df.dropna(subset=['question', 'level'], inplace=True)
    df['level'] = df['level'].str.lower().str.strip()
    df.drop(df[~df['level'].isin(LABEL_TO_ID.keys())].index, inplace=True)
    print(f'  {name}: {len(df)}/{before}')

print('\nLabel distribution in train:')
print(train_df['level'].value_counts())

In [ ]:
# ---- Cell 3: Dataset + loaders (identical to baseline) -------------------
tokenizer = DistilBertTokenizerFast.from_pretrained(Config['model_name'])

class BloomDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts  = df['question'].astype(str).tolist()
        self.labels = [LABEL_TO_ID[l] for l in df['level']]
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True,
                             padding='max_length', max_length=self.max_len,
                             return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = BloomDataset(train_df, tokenizer, Config['max_len'])
val_ds   = BloomDataset(val_df,   tokenizer, Config['max_len'])
test_ds  = BloomDataset(test_df,  tokenizer, Config['max_len'])

train_loader = DataLoader(train_ds, batch_size=Config['batch_size'], shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=Config['batch_size'], shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=Config['batch_size'], shuffle=False,
                          num_workers=2, pin_memory=True)

In [ ]:
# ---- Cell 4: Custom model with mean-pooling + dropout head ---------------
# We replace DistilBertForSequenceClassification with a tiny custom wrapper
# so we can choose pooling strategy and dropout rate. The encoder weights
# are still loaded from the standard pretrained distilbert-base-uncased.

class DistilBertBloomClassifier(nn.Module):
    def __init__(self, model_name: str, num_classes: int,
                 pooling: str = 'mean', head_dropout: float = 0.1):
        super().__init__()
        self.encoder = DistilBertModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size                      # 768
        self.pooling = pooling
        self.pre_classifier = nn.Linear(hidden, hidden)
        self.dropout = nn.Dropout(head_dropout)
        self.classifier = nn.Linear(hidden, num_classes)
        # init head weights (matches HF's default for DistilBertForSequenceClassification)
        nn.init.xavier_uniform_(self.pre_classifier.weight)
        nn.init.zeros_(self.pre_classifier.bias)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = out.last_hidden_state           # (B, T, H)
        if self.pooling == 'cls':
            pooled = hidden_states[:, 0]                 # (B, H)
        else:  # mean-pool, masked
            mask = attention_mask.unsqueeze(-1).float()  # (B, T, 1)
            summed = (hidden_states * mask).sum(dim=1)   # (B, H)
            denom = mask.sum(dim=1).clamp(min=1e-9)      # (B, 1)
            pooled = summed / denom
        x = self.pre_classifier(pooled)
        x = F.gelu(x)
        x = self.dropout(x)
        logits = self.classifier(x)
        return logits


def build_model(unfreeze_top: int):
    model = DistilBertBloomClassifier(
        model_name=Config['model_name'],
        num_classes=Config['num_classes'],
        pooling=Config['pooling'],
        head_dropout=Config['head_dropout'],
    )
    # Freeze encoder layers 0..(6-unfreeze_top-1); head is always trainable.
    for p in model.encoder.parameters():
        p.requires_grad = False
    layers = model.encoder.transformer.layer
    for i in range(6 - unfreeze_top, 6):
        for p in layers[i].parameters():
            p.requires_grad = True
    for p in model.pre_classifier.parameters(): p.requires_grad = True
    for p in model.classifier.parameters():     p.requires_grad = True
    return model.to(device)

_m = build_model(Config['unfreeze_top'])
total   = sum(p.numel() for p in _m.parameters())
train_p = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Total: {total:,}   Trainable: {train_p:,}   ({100*train_p/total:.1f}%)')
print(f'Pooling: {Config["pooling"]}   head_dropout: {Config["head_dropout"]}')
del _m; torch.cuda.empty_cache()

In [ ]:
# ---- Cell 5: Focal loss with class weights + label smoothing -------------
class FocalCrossEntropyLoss(nn.Module):
    """Focal loss (Lin et al. 2017) with class weighting and label smoothing.

    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    With label smoothing eps, the target distribution becomes
    (1-eps) on true class + eps/(K-1) on the others. We compute the loss
    against that soft target with the focal modulator applied.
    """
    def __init__(self, gamma: float = 2.0, alpha: torch.Tensor = None,
                 label_smoothing: float = 0.0, num_classes: int = 3):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha          # (K,) tensor of class weights, or None
        self.eps   = label_smoothing
        self.K     = num_classes

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=-1)        # (B, K)
        probs     = log_probs.exp()                       # (B, K)
        # Soft targets with label smoothing
        with torch.no_grad():
            true_oh = F.one_hot(targets, num_classes=self.K).float()
            if self.eps > 0:
                soft = true_oh * (1.0 - self.eps) + (1.0 - true_oh) * (self.eps / (self.K - 1))
            else:
                soft = true_oh
        # Focal modulator: (1 - p_t)^gamma. p_t is prob assigned to TRUE class.
        p_t = (probs * true_oh).sum(dim=-1)               # (B,)
        focal = (1.0 - p_t).clamp(min=1e-9).pow(self.gamma)  # (B,)
        # Per-class alpha weighting (applied via the true class)
        if self.alpha is not None:
            alpha_t = self.alpha[targets]                  # (B,)
        else:
            alpha_t = torch.ones_like(p_t)
        # CE against soft target, with focal modulator
        ce = -(soft * log_probs).sum(dim=-1)              # (B,)
        loss = alpha_t * focal * ce
        return loss.mean()

# Class weights (same formula as baseline)
counts = train_df['level'].value_counts()
tot = counts.sum()
weights = {l: tot / (Config['num_classes'] * counts[l]) for l in counts.index}
class_weights = torch.tensor(
    [weights[ID_TO_LABEL[i]] for i in range(Config['num_classes'])],
    dtype=torch.float, device=device,
)
print('Class weights:', {ID_TO_LABEL[i]: round(class_weights[i].item(), 3) for i in range(3)})
print(f'Focal gamma: {Config["focal_gamma"]}   Label smoothing: {Config["label_smoothing"]}')

In [ ]:
# ---- Cell 6: LLRD parameter groups (same logic, adapted for new model) ---
def build_llrd_parameter_groups(model, base_lr: float, decay: float, weight_decay: float):
    no_decay_names = ('bias', 'LayerNorm.weight', 'LayerNorm.bias')
    groups = []
    for layer_idx in range(6):
        depth_from_top = 5 - layer_idx
        lr = base_lr * (decay ** depth_from_top)
        prefix = f'encoder.transformer.layer.{layer_idx}.'
        decay_params, nodecay_params = [], []
        for n, p in model.named_parameters():
            if not p.requires_grad or not n.startswith(prefix):
                continue
            (nodecay_params if any(nd in n for nd in no_decay_names) else decay_params).append(p)
        if decay_params:
            groups.append({'params': decay_params, 'lr': lr, 'weight_decay': weight_decay})
        if nodecay_params:
            groups.append({'params': nodecay_params, 'lr': lr, 'weight_decay': 0.0})
    # Head
    head_decay, head_nodecay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if 'classifier' not in n and 'pre_classifier' not in n:
            continue
        (head_nodecay if any(nd in n for nd in no_decay_names) else head_decay).append(p)
    if head_decay:
        groups.append({'params': head_decay, 'lr': base_lr, 'weight_decay': weight_decay})
    if head_nodecay:
        groups.append({'params': head_nodecay, 'lr': base_lr, 'weight_decay': 0.0})
    return groups

In [ ]:
# ---- Cell 7: Train / eval loops (slightly adapted for custom model) ------
def train_one_epoch(model, loader, optimizer, scheduler, loss_fn, scaler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        optimizer.zero_grad()
        ids   = batch['input_ids'].to(device, non_blocking=True)
        attn  = batch['attention_mask'].to(device, non_blocking=True)
        lab   = batch['labels'].to(device, non_blocking=True)
        if Config['use_amp']:
            with autocast():
                logits = model(ids, attn)
                loss = loss_fn(logits, lab)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            logits = model(ids, attn)
            loss = loss_fn(logits, lab)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0; preds, gold = [], []
    for batch in loader:
        ids  = batch['input_ids'].to(device, non_blocking=True)
        attn = batch['attention_mask'].to(device, non_blocking=True)
        lab  = batch['labels'].to(device, non_blocking=True)
        if Config['use_amp']:
            with autocast():
                logits = model(ids, attn)
        else:
            logits = model(ids, attn)
        total_loss += loss_fn(logits.float(), lab).item()
        preds.extend(logits.argmax(dim=-1).cpu().tolist())
        gold.extend(lab.cpu().tolist())
    return total_loss / len(loader), f1_score(gold, preds, average='macro'), preds, gold

In [ ]:
# ---- Cell 8: MAIN tuned training run -------------------------------------
model = build_model(Config['unfreeze_top'])
loss_fn = FocalCrossEntropyLoss(
    gamma=Config['focal_gamma'],
    alpha=class_weights,
    label_smoothing=Config['label_smoothing'],
    num_classes=Config['num_classes'],
)
optimizer = torch.optim.AdamW(
    build_llrd_parameter_groups(model, Config['lr'], Config['llrd_decay'], Config['weight_decay'])
)
total_steps  = len(train_loader) * Config['epochs']
warmup_steps = int(total_steps * Config['warmup_frac'])
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps,
)
scaler = GradScaler(enabled=Config['use_amp'])

history = []
best_f1, best_state = -1.0, None
for epoch in range(1, Config['epochs'] + 1):
    tr_loss = train_one_epoch(model, train_loader, optimizer, scheduler, loss_fn, scaler)
    val_loss, val_f1, _, _ = evaluate(model, val_loader, loss_fn)
    history.append({'epoch': epoch, 'train_loss': tr_loss, 'val_loss': val_loss, 'val_macro_f1': val_f1})
    print(f'  epoch {epoch}/{Config["epochs"]}   train={tr_loss:.4f}  val={val_loss:.4f}  val_f1={val_f1:.4f}')
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
test_loss, test_f1, preds, gold = evaluate(model, test_loader, loss_fn)
print(f'\nTEST  macro_f1={test_f1:.4f}   loss={test_loss:.4f}')
print(classification_report(gold, preds, target_names=['easy','medium','hard'], digits=4))
print('Confusion (rows=true, cols=pred):  easy / medium / hard')
print(confusion_matrix(gold, preds))

In [ ]:
# ---- Cell 9: Save HF-compatible weights + custom state + REPORT ARTIFACTS

# (a) HF-compatible save so BloomClassifier.load() can read it.
#     NOTE: HF DistilBertForSequenceClassification uses [CLS] pooling internally.
#     If your tuned model used mean-pooling, predictions through the HF format
#     will differ slightly from the in-notebook results. The custom state dict
#     (saved below) preserves mean-pool exactly.
from transformers import DistilBertForSequenceClassification
hf_model = DistilBertForSequenceClassification.from_pretrained(
    Config['model_name'], num_labels=Config['num_classes'],
    seq_classif_dropout=Config['head_dropout'],
)
hf_model.distilbert.load_state_dict(model.encoder.state_dict())
hf_model.pre_classifier.weight.data.copy_(model.pre_classifier.weight.data)
hf_model.pre_classifier.bias.data.copy_(model.pre_classifier.bias.data)
hf_model.classifier.weight.data.copy_(model.classifier.weight.data)
hf_model.classifier.bias.data.copy_(model.classifier.bias.data)

hf_model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
with open(f'{OUT_DIR}/label_map.json', 'w') as f:
    json.dump(LABEL_TO_ID, f, indent=2)
with open(f'{OUT_DIR}/training_history.json', 'w') as f:
    json.dump({'history': history, 'test_macro_f1': test_f1,
               'test_loss': test_loss, 'config': Config}, f, indent=2)

# (b) Custom state dict (preserves mean-pool exactly)
torch.save({
    'state_dict': model.state_dict(),
    'config': Config,
    'test_macro_f1': test_f1,
}, f'{OUT_DIR}/tuned_model_state.pt')

# (c) Report artifacts (confusion matrix + per-question predictions)
target_names = ['easy', 'medium', 'hard']
text_report = classification_report(gold, preds, target_names=target_names, digits=4)
with open(f'{OUT_DIR}/test_classification_report.txt', 'w') as f:
    f.write(f'TEST macro_f1 = {test_f1:.4f}   loss = {test_loss:.4f}\n\n')
    f.write(text_report)
with open(f'{OUT_DIR}/test_classification_report.json', 'w') as f:
    json.dump(classification_report(gold, preds, target_names=target_names,
                                    digits=4, output_dict=True), f, indent=2)
cm = confusion_matrix(gold, preds)
with open(f'{OUT_DIR}/test_confusion_matrix.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['true \\ pred'] + target_names)
    for i, row in enumerate(cm):
        w.writerow([target_names[i]] + list(row))
with open(f'{OUT_DIR}/test_predictions.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['idx', 'question', 'true_level', 'predicted_level', 'correct', 'source'])
    qs  = test_df['question'].tolist()
    src = test_df['source'].tolist() if 'source' in test_df.columns else [''] * len(gold)
    for i, (q, t, p) in enumerate(zip(qs, gold, preds)):
        w.writerow([i, q, target_names[t], target_names[p],
                    'yes' if t == p else 'no',
                    src[i] if i < len(src) else ''])

print(f'Saved to {OUT_DIR}')
print(f'  - HF weights:               model.safetensors, config.json, tokenizer files')
print(f'  - Custom state dict:        tuned_model_state.pt  (preserves mean-pool)')
print(f'  - training_history.json     per-epoch curves')
print(f'  - test_classification_report.txt + .json')
print(f'  - test_confusion_matrix.csv')
print(f'  - test_predictions.csv      per-question predictions for error analysis')

In [ ]:
# ---- Cell 10: Sanity check (uses the TUNED mean-pool model) -------------
model.eval()
samples = [
    'Define a semaphore.',
    'Compute the number of page faults for the reference string 1,2,3,4,1,2,5 under LRU with 3 frames.',
    'Design a deadlock-free dining philosophers protocol using only binary semaphores and justify why it avoids circular wait.',
]
with torch.no_grad():
    enc = tokenizer(samples, padding=True, truncation=True,
                    max_length=Config['max_len'], return_tensors='pt').to(device)
    logits = model(enc['input_ids'], enc['attention_mask'])
    probs  = torch.softmax(logits, dim=-1).cpu().numpy()
    preds  = logits.argmax(dim=-1).cpu().tolist()
for s, p, pr in zip(samples, preds, probs):
    print(f'\n[{ID_TO_LABEL[p]}]  (easy={pr[0]:.2f}  medium={pr[1]:.2f}  hard={pr[2]:.2f})')
    print(f'  Q: {s}')

print(f'\n\n=== TUNED vs BASELINE comparison ===')
print(f'Baseline (3-layer, plain CE, [CLS] pool, 4 epochs):  test macro-F1 = 0.7261')
print(f'Tuned    (3-layer, focal+smoothing, mean-pool, 6e):  test macro-F1 = {test_f1:.4f}')
print(f'Delta:                                                 {test_f1 - 0.7261:+.4f}')

In [ ]:
# ---- Final cell: ZIP the output for reliable single-file download -------
import shutil, os
zip_path = shutil.make_archive('/kaggle/working/bloom_distilbert_tuned', 'zip', OUT_DIR)
sz_mb = os.path.getsize(zip_path) / 1e6
print(f'Created: {zip_path}  ({sz_mb:.1f} MB)')
print(f'\nDownload bloom_distilbert_tuned.zip from the right-side Output panel,')
print(f'then unzip locally to question-generation-module/models/bloom_distilbert/')